# SQL — Introdução e criação de tabelas

Este material apresenta os fundamentos iniciais de SQL utilizando PostgreSQL. O objetivo é construir uma pequena base de dados do zero, começando pelo ambiente Docker e avançando para a criação de tabelas, restrições e chaves simples e compostas.

Neste primeiro material serão abordados:

- execução do PostgreSQL com Docker;
- conexão com o banco de dados;
- `CREATE DATABASE` (conceitualmente);
- `CREATE TABLE`;
- tipos de dados básicos;
- `PRIMARY KEY`;
- `FOREIGN KEY`;
- `NOT NULL`;
- `UNIQUE`;
- `CHECK`;
- `DEFAULT`;
- chaves compostas;
- relacionamento entre tabelas.


## Preparando o ambiente

Para os exemplos, utilizaremos PostgreSQL executando em um container Docker.

A ideia é manter o ambiente simples: o Docker será responsável apenas por fornecer o servidor de banco de dados. Os comandos SQL serão executados posteriormente através de uma ferramenta como DBeaver ou `psql`.


### Arquivos do ambiente

Estrutura sugerida:

```text
sql/
├── docker-compose.yml
└── .env
```


### Arquivo `.env`

O arquivo `.env` concentra as informações utilizadas pelo container para configurar o PostgreSQL.

```env
POSTGRES_USER=sql_user
POSTGRES_PASSWORD=sql_password
POSTGRES_DB=sql_bootcamp
```

Em um ambiente real, senhas não devem ser colocadas diretamente em arquivos que serão versionados. Para um laboratório didático, porém, essa abordagem é suficiente.

### Arquivo `docker-compose.yml`

```yaml
services:
  postgres:
    image: postgres:18
    container_name: sql-postgres
    restart: unless-stopped

    env_file:
      - .env

    ports:
      - "5432:5432"
```

Não utilizaremos um volume neste laboratório. Dessa forma, o ambiente pode ser recriado facilmente quando necessário.

### Subindo o PostgreSQL

No diretório que contém o `docker-compose.yml`:

```bash
docker compose up -d
```

Podemos verificar se o container está executando:

```bash
docker compose ps
```

## Conectando ao banco

Podemos utilizar uma ferramenta gráfica como o DBeaver.

Parâmetros da conexão:

| Parâmetro | Valor |
|---|---|
| Host | `localhost` |
| Port | `5432` |
| Database | `sql_bootcamp` |
| Username | `sql_user` |
| Password | `sql_password` |

# SQL e tabelas

Uma tabela representa uma estrutura organizada de dados. Ela é formada por **colunas**, que definem quais informações serão armazenadas, e **linhas**, que representam os registros.

Por exemplo, uma tabela de clientes pode possuir:

| id | nome | email |
|---:|---|---|
| 1 | Ana | ana@email.com |
| 2 | Carlos | carlos@email.com |

Antes de inserir registros, precisamos definir a estrutura da tabela.

## CREATE TABLE

O comando `CREATE TABLE` cria uma nova tabela.

Estrutura básica:

```sql
CREATE TABLE nome_tabela (
    coluna1 tipo,
    coluna2 tipo,
    coluna3 tipo
);
```

Um primeiro exemplo:

```sql
CREATE TABLE clientes (
    id INTEGER,
    nome VARCHAR(100),
    email VARCHAR(150)
);
```

Nesse exemplo, criamos três colunas:

- `id`: número inteiro;
- `nome`: texto com até 100 caracteres;
- `email`: texto com até 150 caracteres.

Embora essa tabela seja válida, ainda não estamos impondo regras importantes sobre os dados.

## Tipos de dados básicos

Alguns tipos bastante utilizados no PostgreSQL:

| Tipo | Utilização |
|---|---|
| `INTEGER` | números inteiros |
| `BIGINT` | números inteiros maiores |
| `NUMERIC(10,2)` | valores numéricos com precisão definida |
| `VARCHAR(100)` | textos com tamanho máximo |
| `TEXT` | textos sem limite definido de tamanho prático |
| `BOOLEAN` | `TRUE` ou `FALSE` |
| `DATE` | datas |
| `TIMESTAMP` | data e horário |

A escolha do tipo deve refletir a natureza do dado que será armazenado.

## PRIMARY KEY

A **chave primária** identifica unicamente cada registro de uma tabela.

Uma tabela de clientes, por exemplo, pode utilizar `id` como chave primária:

```sql
CREATE TABLE clientes (
    id INTEGER PRIMARY KEY,
    nome VARCHAR(100),
    email VARCHAR(150)
);
```

Com isso, dois registros não podem possuir o mesmo `id`, e o `id` não pode ser `NULL`.

### Gerando identificadores automaticamente

Uma forma comum no PostgreSQL é utilizar uma coluna `IDENTITY`:

```sql
CREATE TABLE clientes (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100),
    email VARCHAR(150)
);
```

Agora o banco gera automaticamente o identificador quando um novo cliente é inserido.

## NOT NULL

`NOT NULL` determina que uma coluna obrigatoriamente deve possuir um valor.

```sql
CREATE TABLE clientes (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    email VARCHAR(150) NOT NULL
);
```

Nesse caso, não podemos criar um cliente sem informar `nome` e `email`.

## UNIQUE

`UNIQUE` impede que determinado valor seja repetido na coluna.

Por exemplo, podemos exigir que cada cliente tenha um e-mail diferente:

```sql
CREATE TABLE clientes (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    email VARCHAR(150) NOT NULL UNIQUE
);
```

A diferença conceitual é importante: `PRIMARY KEY` identifica o registro, enquanto `UNIQUE` estabelece uma regra de unicidade para um valor.

## CHECK

`CHECK` permite definir uma condição que os valores devem satisfazer.

Exemplo:

```sql
CREATE TABLE produtos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    preco NUMERIC(10,2) NOT NULL,
    estoque INTEGER NOT NULL,

    CHECK (preco >= 0),
    CHECK (estoque >= 0)
);
```

O banco passa a impedir valores negativos para `preco` e `estoque`.

## DEFAULT

`DEFAULT` define um valor utilizado quando nenhum valor é informado para determinada coluna.

```sql
CREATE TABLE produtos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    preco NUMERIC(10,2) NOT NULL CHECK (preco >= 0),
    estoque INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0)
);
```

Se um produto for criado sem informar o estoque, o banco utilizará `0`.

# FOREIGN KEY

Uma `FOREIGN KEY` representa uma referência entre tabelas.

Imagine que cada pedido pertence a um cliente. Podemos criar primeiro a tabela `clientes`:

```sql
CREATE TABLE clientes (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    email VARCHAR(150) NOT NULL UNIQUE
);
```

E então criar `pedidos`:

```sql
CREATE TABLE pedidos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    cliente_id INTEGER NOT NULL,
    data_pedido DATE NOT NULL,

    FOREIGN KEY (cliente_id) REFERENCES clientes(id)
);
```

Agora `cliente_id` precisa corresponder a um `id` existente em `clientes`.

## Definindo a FOREIGN KEY diretamente na coluna

Para uma chave estrangeira simples, também podemos escrever:

```sql
CREATE TABLE pedidos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    cliente_id INTEGER NOT NULL REFERENCES clientes(id),
    data_pedido DATE NOT NULL
);
```

As duas formas são válidas. A forma com `FOREIGN KEY (...) REFERENCES ...` costuma ser mais explícita e fica especialmente útil quando começamos a trabalhar com múltiplas colunas.

# Chaves compostas

Uma chave composta utiliza **duas ou mais colunas em conjunto** para identificar unicamente um registro.

Um exemplo clássico é uma tabela que representa quais produtos fazem parte de cada pedido.

Um mesmo produto pode aparecer em vários pedidos, e um pedido pode possuir vários produtos. A combinação:

```text
pedido_id + produto_id
```

pode identificar unicamente cada item do pedido.

### Criando uma chave primária composta

```sql
CREATE TABLE itens_pedido (
    pedido_id INTEGER NOT NULL,
    produto_id INTEGER NOT NULL,
    quantidade INTEGER NOT NULL,
    preco_unitario NUMERIC(10,2) NOT NULL,

    PRIMARY KEY (pedido_id, produto_id),

    FOREIGN KEY (pedido_id) REFERENCES pedidos(id),
    FOREIGN KEY (produto_id) REFERENCES produtos(id),

    CHECK (quantidade > 0),
    CHECK (preco_unitario >= 0)
);
```

A combinação das duas colunas é a chave primária. Isso significa que podemos ter:

```text
pedido_id | produto_id
----------+-----------
1         | 10
1         | 20
2         | 10
2         | 30
```

Mas não podemos inserir novamente `(1, 10)`, pois essa combinação já existe.

## Exemplo completo

Podemos juntar os conceitos apresentados em um pequeno modelo de vendas.

### Clientes

```sql
CREATE TABLE clientes (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    email VARCHAR(150) NOT NULL UNIQUE
);
```

### Produtos

```sql
CREATE TABLE produtos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    preco NUMERIC(10,2) NOT NULL CHECK (preco >= 0),
    estoque INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0)
);
```

### Pedidos

```sql
CREATE TABLE pedidos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    cliente_id INTEGER NOT NULL,
    data_pedido DATE NOT NULL,

    FOREIGN KEY (cliente_id) REFERENCES clientes(id)
);
```

### Itens do pedido

```sql
CREATE TABLE itens_pedido (
    pedido_id INTEGER NOT NULL,
    produto_id INTEGER NOT NULL,
    quantidade INTEGER NOT NULL CHECK (quantidade > 0),
    preco_unitario NUMERIC(10,2) NOT NULL CHECK (preco_unitario >= 0),

    PRIMARY KEY (pedido_id, produto_id),

    FOREIGN KEY (pedido_id) REFERENCES pedidos(id),
    FOREIGN KEY (produto_id) REFERENCES produtos(id)
);
```


## Ordem de criação das tabelas

Quando existem relacionamentos entre tabelas, a ordem de criação importa.

No nosso exemplo:

```text
clientes
   │
   └──> pedidos
            │
            └──> itens_pedido <── produtos
```

Por isso, devemos criar as tabelas nesta ordem:

1. `clientes`
2. `produtos`
3. `pedidos`
4. `itens_pedido`

A tabela `itens_pedido`, por exemplo, depende de `pedidos` e `produtos` porque possui chaves estrangeiras para essas duas tabelas.